In [27]:
import torch
from transformers import Wav2Vec2Model,Wav2Vec2Processor

从hugging face上下载权重

In [7]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="facebook/wav2vec2-large-960h-lv60-self",local_dir="./wav2vec2_model_weights")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/4.47k [00:00<?, ?B/s]

'D:\\研一学习\\code\\diffusers_use\\model_weights'

In [28]:
model=Wav2Vec2Model.from_pretrained("./wav2vec2_model_weights",torch_dtype=torch.float16).to("cuda:0")

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at ./wav2vec2_model_weights and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
print(model)

Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2LayerNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (1-4): 4 x Wav2Vec2LayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2LayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=1024, bias=True)
    (dropout)

In [30]:
!pip install soundfile

In [31]:
import soundfile as sf
audio_path="audio_clip.mp3"
audio_input,sample_rate=sf.read(audio_path)

In [32]:
print(type(audio_input))
print(audio_input.size)

print(type(sample_rate))
print(sample_rate)

<class 'numpy.ndarray'>
463104
<class 'int'>
44100


In [33]:
processsor=Wav2Vec2Processor.from_pretrained("./wav2vec2_model_weights")

In [35]:
input_values=processsor(audio_input,return_tensors="pt").input_values

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


In [37]:
print(type(input_values))
print(input_values.shape)

<class 'torch.Tensor'>
torch.Size([1, 463104])


In [41]:
input_values=torch.tensor(input_values,dtype=torch.float16).to("cuda:0")
out_put=model(input_values)

C:\Users\Shipu\AppData\Local\Temp\ipykernel_16904\2455904956.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_values=torch.tensor(input_values,dtype=torch.float16).to("cuda:0")


In [46]:
print(out_put["last_hidden_state"].shape)

torch.Size([1, 1446, 1024])
